# Chapter 9 - Video-Language Models

In [ ]:
# Here some installs that we will be using in multiple parts of the chapter
!pip -q install -U transformers==5.2.0
!pip -q install -U torchcodec huggingface_hub
!pip -q install -U decord av qwen_vl_utils num2words faiss-cpu datasets peft bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 120.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 179.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 MB 76.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 143.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.7/520.7 kB 58.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 64.5 MB/s eta 0:00:00


In [ ]:
# Very important to be logged in!
from huggingface_hub import login
login()


In [ ]:
# We could compact all the imports in this single cell, remember to run it before going through the different sections of the notebook
from huggingface_hub import hf_hub_download
from transformers import pipeline, AutoProcessor, AutoModelForImageTextToText, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
import numpy as np
from typing import Dict, List, Tuple
from pathlib import Path
from datasets import load_dataset

import faiss
import torch
import importlib.util
import sys
import os
import subprocess


In [ ]:
# We will be using those two videos as examples across several part of the notebook
spaghetti_mp4 = hf_hub_download("vlmbook/videos", "eating_spaghetti.mp4", repo_type="dataset")
minecraft_mp4  = hf_hub_download("vlmbook/videos", "09KmKSz4r_Y.mp4", repo_type="dataset")


eating_spaghetti.mp4:   0%|          | 0.00/1.01M [00:00<?, ?B/s]

09KmKSz4r_Y.mp4:   0%|          | 0.00/2.73M [00:00<?, ?B/s]

# 9.1 Foundations
## 9.1.1 Video-Language tasks

### Video Classification

In [ ]:

clf = pipeline(
   "video-classification",
   model="MCG-NJU/videomae-small-finetuned-kinetics",
   device=0,
   dtype=torch.bfloat16,
)
print(clf(spaghetti_mp4, top_k=5))


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/88.2M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/186 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/88.2M [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

The image processor of type `VideoMAEImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
`use_fast` is set to `True` but the image processor class does not have a fast version.  Falling back to the slow version.


[{'score': 0.765625, 'label': 'eating spaghetti'}, {'score': 0.0185546875, 'label': 'eating burger'}, {'score': 0.015625, 'label': 'feeding fish'}, {'score': 0.00933837890625, 'label': 'eating ice cream'}, {'score': 0.009033203125, 'label': 'making pizza'}]


### Text-to-Video Retrieval

In [ ]:
from huggingface_hub import hf_hub_download
import importlib.util
import sys
import torch

# Temporary patch!!
# While we wait for this PR https://huggingface.co/Qwen/Qwen3-VL-Embedding-2B/discussions/19
# to merge, you can run the code with the patch:

import transformers.utils.generic as t_generic

if not hasattr(t_generic, "check_model_inputs"):
    def check_model_inputs(fn=None, *args, **kwargs):
        # Suporta tant @check_model_inputs com @check_model_inputs()
        if fn is None:
            def deco(f):
                return f
            return deco
        return fn

    t_generic.check_model_inputs = check_model_inputs
# End of temporary patch


# Qwen3-VL-Embedding ships a helper script in its model repo.
# We download it and import the Qwen3VLEmbedder class directly.

script_path = hf_hub_download(
    repo_id="Qwen/Qwen3-VL-Embedding-2B",
    filename="scripts/qwen3_vl_embedding.py",
)
spec = importlib.util.spec_from_file_location("qwen3_vl_embedding", script_path)
mod  = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
sys.modules['qwen3_vl_embedding'] = mod
Qwen3VLEmbedder = mod.Qwen3VLEmbedder
embedder = Qwen3VLEmbedder(
    model_name_or_path="Qwen/Qwen3-VL-Embedding-2B",
    torch_dtype=torch.bfloat16
)

qwen3_vl_embedding.py: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/4.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/817 [00:00<?, ?B/s]

In [ ]:
videos = [spaghetti_mp4, minecraft_mp4]
# Embed one text query and two videos in a single call.
# The embedder returns L2-normalized vectors, so dot product = cosine similarity.
inputs = [
    {"text": "a person eating spaghetti"},
    {"video": videos[0], "fps": 1.0, "max_frames": 32},
    {"video": videos[1], "fps": 1.0, "max_frames": 32},
]
emb = embedder.process(inputs)  # (3, D) tensor, L2-normalized
q = emb[0:1]   # query embedding   (1, D)
V = emb[1:]    # video embeddings  (2, D)
scores = (V @ q.T).squeeze(1)
rank   = scores.argsort(descending=True).tolist()
print("Query:", inputs[0]["text"])
for i in rank:
    print(f"  score={scores[i].item():.3f}  video={videos[i].split("/")[-1]}")

qwen-vl-utils using torchcodec to read video.


Query: a person eating spaghetti
  score=0.750  video=eating_spaghetti.mp4
  score=0.164  video=09KmKSz4r_Y.mp4


## Captioning, Summarization, and Video QA


In [ ]:
model_id  = "HuggingFaceTB/SmolVLM2-500M-Video-Instruct"
processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    dtype=torch.bfloat16,
    device_map="auto"
).eval()


processor_config.json:   0%|          | 0.00/67.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/430 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/868 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.03G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/489 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

In [ ]:

def ask_video(video_path: str, prompt: str, num_frames: int = 8) -> str:
    messages = [{
       "role": "user",
        "content": [
            {"type": "video", "path": video_path},
            {"type": "text",  "text": prompt},
        ],
    }]
    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
        do_sample_frames=True,
        num_frames=num_frames,
    ).to(model.device)
    if "pixel_values_videos" in inputs:
        inputs["pixel_values_videos"] = inputs["pixel_values_videos"].to(dtype)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=128, do_sample=False)

    prompt_len = inputs["input_ids"].shape[-1]
    return processor.batch_decode(
        out[:, prompt_len:], skip_special_tokens=True
    )[0].strip()


# Captioning
print("CAPTION:\n", ask_video(minecraft_mp4, "Write a one-sentence caption for this video."))

# Summarization
print("\nSUMMARY:\n", ask_video(minecraft_mp4, "Summarize the video in 3 bullet points."))

# Video QA
print("\nQA:\n", ask_video(spaghetti_mp4, "What is the person doing? Answer briefly."))

You have used fast image processor with LANCZOS resample which not yet supported for torch.Tensor. BICUBIC resample will be used as an alternative. Please fall back to image processor if you want full consistency with the original model.


CAPTION:
 The video showcases a Minecraft character in a dark, stone-walled room, performing various actions such as jumping, running, and shooting, with the player's inventory visible in the bottom right corner.

SUMMARY:
 The video shows a character in a room with a stone wall and a glowing blue block, followed by a character in a blue outfit with a red mask and a yellow sword, and then a character in a black outfit with a red mask and a yellow sword.

QA:
 Preparing food.


# 9.3.2 Retrieval Pipelines That Scale


### Step 1: Segment videos
We prepare a function to split videos in small segments

In [ ]:
def segment_video(
    video_path: str, out_dir: str, segment_seconds: int = 5
) -> List[str]:
    """
    Split a video into fixed-length MP4 segments using ffmpeg.
    Returns sorted list of segment file paths.
    """
    out_dir_path = Path(out_dir).resolve()
    out_dir_path.mkdir(parents=True, exist_ok=True)
    pattern = str(out_dir_path / "seg_%05d.mp4")

    subprocess.run(
        [
            "ffmpeg", "-y",
            "-i", video_path,
            "-c", "copy",            # stream copy — fast, no re-encoding
            "-map", "0",
            "-f", "segment",
            "-segment_time", str(segment_seconds),
            "-reset_timestamps", "1",
            pattern,
        ],
        check=True,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    # Return absolute paths for the segments
    return sorted(str(p.resolve()) for p in out_dir_path.glob("seg_*.mp4"))

In [ ]:
# We get the Qwen3-VL Embedding model again and we prepare the functions to embed text/video
script_path = hf_hub_download(
    repo_id="Qwen/Qwen3-VL-Embedding-2B",
    filename="scripts/qwen3_vl_embedding.py",
)
spec = importlib.util.spec_from_file_location("qwen3_vl_embedding", script_path)
mod  = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
sys.modules['qwen3_vl_embedding'] = mod  # Explicitly add the module to sys.modules
Qwen3VLEmbedder = mod.Qwen3VLEmbedder
embedder = Qwen3VLEmbedder(
    model_name_or_path="Qwen/Qwen3-VL-Embedding-2B",
    torch_dtype=torch.float16
)

# Embedding helpers
def embed_text(text: str) -> np.ndarray:
    """Embed a text query. Returns (D,) numpy array, L2-normalized."""
    emb = embedder.process([{"text": text}])
    return emb[0].detach().cpu().float().numpy()

def embed_video_segment(path: str, fps: float = 1.0, max_frames: int = 32) -> np.ndarray:
    """Embed a video segment. Returns (D,) numpy array, L2-normalized."""
    emb = embedder.process([{"video": path, "fps": fps, "max_frames": max_frames}])
    return emb[0].detach().cpu().float().numpy()


Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

In [ ]:

#Segment, embed, and index our demo videos

all_segments: List[str] = []
segment_meta: List[Dict] = []

for vid in [spaghetti_mp4, minecraft_mp4]:
    out_dir = Path("video_segments") / Path(vid).stem
    segs = segment_video(vid, str(out_dir), segment_seconds=5)
    for seg in segs:
        all_segments.append(seg)
        segment_meta.append({"source_video": vid, "segment_path": seg})

print(f"Total segments: {len(all_segments)}")



Total segments: 11


In [ ]:

# Embed all segments (offline step — do this once)
vectors = np.stack(
    [embed_video_segment(seg) for seg in all_segments], axis=0
)  # (N, D)


In [ ]:

# Build FAISS index. Since embeddings are L2-normalized,
# inner product (IndexFlatIP) equals cosine similarity.
index = faiss.IndexFlatIP(vectors.shape[1])
index.add(vectors.astype(np.float32))


In [ ]:

# Query function
def search_segments(query: str, k: int = 5) -> List[Tuple[float, Dict]]:
    """Retrieve top-k segments for a text query."""
    q = embed_text(query)[None, :]  # (1, D)
    scores, ids = index.search(q.astype(np.float32), k)
    return [
        (float(s), segment_meta[i])
        for s, i in zip(scores[0], ids[0])
        if i >= 0
    ]

# Try it
for q in ["someone eating pasta", "a videogame"]:
    print(f"\nQuery: {q}")
    for score, meta in search_segments(q, k=3):
        print(f"  score={score:.3f}  seg={meta['segment_path']}")


Query: someone eating pasta
  score=0.729  seg=/content/video_segments/eating_spaghetti/seg_00000.mp4
  score=0.507  seg=/content/video_segments/eating_spaghetti/seg_00001.mp4
  score=0.329  seg=/content/video_segments/09KmKSz4r_Y/seg_00003.mp4

Query: a videogame
  score=0.600  seg=/content/video_segments/09KmKSz4r_Y/seg_00003.mp4
  score=0.570  seg=/content/video_segments/09KmKSz4r_Y/seg_00006.mp4
  score=0.562  seg=/content/video_segments/09KmKSz4r_Y/seg_00005.mp4


### Optional: ReRank

In [ ]:
# Download and import the reranker helper (same pattern as the embedder)
rerank_script = hf_hub_download(
    repo_id="Qwen/Qwen3-VL-Reranker-2B",
    filename="scripts/qwen3_vl_reranker.py",
)
spec = importlib.util.spec_from_file_location("qwen3_vl_reranker", rerank_script)
rerank_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(rerank_mod)

reranker = rerank_mod.Qwen3VLReranker(
    model_name_or_path="Qwen/Qwen3-VL-Reranker-2B",
)



qwen3_vl_reranker.py: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/4.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/213 [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/817 [00:00<?, ?B/s]

In [ ]:

def rerank_segments(
    query: str, candidates: List[Dict], fps: float = 1.0, max_frames: int = 32
) -> List[Tuple[float, Dict]]:
    """Re-score retrieved segments with a cross-encoder for higher precision."""
    payload = {
        "query": {"text": query},
        "documents": [{"video": c["segment_path"]} for c in candidates],
        "fps": fps,
        "max_frames": max_frames,
    }
    scores = reranker.process(payload)  # list[float], aligned with candidates
    return sorted(zip(scores, candidates), key=lambda x: x[0], reverse=True)

# Two-stage retrieval
query = "someone eating pasta"

# Stage 1: fast approximate retrieval (embedding + FAISS)
stage1 = search_segments(query, k=10)
candidates = [meta for _, meta in stage1]

# Stage 2: precise reranking (cross-encoder)
ranked = rerank_segments(query, candidates)

print(f"Query: {query}")
print("Top reranked segments:")
for score, meta in ranked[:5]:
    print(f"  rerank_score={score:.3f}  seg={meta['segment_path']}")

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:2299: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


Query: someone eating pasta
Top reranked segments:
  rerank_score=0.645  seg=/content/video_segments/eating_spaghetti/seg_00000.mp4
  rerank_score=0.451  seg=/content/video_segments/eating_spaghetti/seg_00001.mp4
  rerank_score=0.305  seg=/content/video_segments/09KmKSz4r_Y/seg_00005.mp4
  rerank_score=0.299  seg=/content/video_segments/09KmKSz4r_Y/seg_00003.mp4
  rerank_score=0.277  seg=/content/video_segments/09KmKSz4r_Y/seg_00007.mp4


# 9.3.3 Video-RAG: Retrieval-Augmented Video QA

In [ ]:
# We get again our tiny and powerful SmolVLM2
model_id = "HuggingFaceTB/SmolVLM2-500M-Video-Instruct"
rag_processor = AutoProcessor.from_pretrained(model_id)
rag_model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    dtype=torch.bfloat16,
    device_map="auto"
).eval()

def answer_with_segments(
    segment_paths: List[str],
    question: str,
    num_frames: int = 2, # Reduced num_frames further to ensure consistency
    max_new_tokens: int = 128,
) -> str:
    """
    Feed multiple retrieved video segments plus a question to SmolVLM2.
    Each segment becomes a separate {"type": "video"} entry in the
    chat message, so the model sees all of them as context.
    """
    content = [{"type": "video", "path": p} for p in segment_paths]
    content.append({"type": "text", "text": question})

    messages = [{"role": "user", "content": content}]

    inputs = rag_processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
        do_sample_frames=True,
        num_frames=num_frames,
    ).to(rag_model.device)

    if "pixel_values_videos" in inputs and torch.is_floating_point(
        inputs["pixel_values_videos"]
    ):
        inputs["pixel_values_videos"] = inputs["pixel_values_videos"].to(dtype)

    with torch.no_grad():
        out = rag_model.generate(
            **inputs, do_sample=False, max_new_tokens=max_new_tokens
        )
    prompt_len = inputs["input_ids"].shape[-1]
    return rag_processor.batch_decode(
        out[:, prompt_len:], skip_special_tokens=True
    )[0].strip()

Loading weights:   0%|          | 0/489 [00:00<?, ?it/s]

In [ ]:
# Here is the pipeline - first retrieve and then asnwer

def video_rag(question: str, k: int = 5) -> str:
    """
    End-to-end Video-RAG: retrieve relevant segments, then generate
    a grounded answer using only those segments as context.
    """
    # Step 1: Retrieve (fast — embedding + FAISS)
    retrieved = search_segments(question, k=k)
    segment_paths = [meta["segment_path"] for _, meta in retrieved]

    # Step 2: Generate (expensive — but only over k segments, not the whole library)
    prompt = (
        "Answer the question using ONLY the provided video clips. "
        "If the clips do not contain enough information, say so.\n\n"
        f"Question: {question}"
    )
    answer = answer_with_segments(segment_paths, prompt)

    print(f"Question: {question}")
    print(f"\nRetrieved {len(segment_paths)} segments:")
    for score, meta in retrieved:
        print(f"  score={score:.3f}  {meta['segment_path']}")
    print(f"\nAnswer: {answer}")
    return answer

video_rag("Which food do you see in the video?", k=5)


Question: Which food do you see in the video?

Retrieved 5 segments:
  score=0.516  /content/video_segments/eating_spaghetti/seg_00001.mp4
  score=0.477  /content/video_segments/eating_spaghetti/seg_00000.mp4
  score=0.399  /content/video_segments/09KmKSz4r_Y/seg_00003.mp4
  score=0.391  /content/video_segments/09KmKSz4r_Y/seg_00006.mp4
  score=0.386  /content/video_segments/09KmKSz4r_Y/seg_00007.mp4

Answer: spaghetti


'spaghetti'

## 9.3.4 Fine-Tuning a Video Language Model for Your Domain


In [ ]:
# We load a dataset
ds = load_dataset("TIGER-Lab/VideoFeedback", "real")
split_ds = ds["train"].train_test_split(test_size=0.5)
train_ds = split_ds["train"]
del split_ds, ds


README.md: 0.00B [00:00, ?B/s]

real/test-00000-of-00001.parquet:   0%|          | 0.00/35.8k [00:00<?, ?B/s]

real/train-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/80 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/4000 [00:00<?, ? examples/s]

In [ ]:
# Take a sneak peek.

print(f"prompt:  {train_ds[0]['text prompt']}, video: {train_ds[0]['video link']}")

prompt:  There is a bottle of apple cider vinegar sitting on a wooden surface., video: https://huggingface.co/datasets/hexuan21/VideoFeedback-videos-mp4/resolve/main/p/p106007.mp4


In [ ]:
# We will fine-tune SmolVLM2

model_id = "HuggingFaceTB/SmolVLM2-500M-Video-Instruct"
your_model_folder = "smolvlm2-qa-qlora"
tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

# Define the quantization configuration for 4-bit loading
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    quantization_config=quantization_config, # Use quantization_config instead of load_in_4bit
    device_map="auto"
)

model = prepare_model_for_kbit_training(model)
cfg = LoraConfig(
    r=16, lora_alpha=32, target_modules=["q_proj","k_proj","v_proj","gate_proj"],
    lora_dropout=0.05, task_type="CAUSAL_LM"
)
model = get_peft_model(model, cfg)


Loading weights:   0%|          | 0/489 [00:00<?, ?it/s]

In [ ]:
def collate(batch):
    all_conversations = []
    for ex in batch:
        # Ensure required keys exist before trying to access them
        if "video link" not in ex or "text prompt" not in ex or "conversations" not in ex or len(ex["conversations"]) < 2:
            print(f"Skipping example due to missing or incomplete data: {ex.get('id', 'N/A')}")
            continue

        video_path = ex["video link"]
        question = ex["text prompt"]
        answer = ex["conversations"][1]["value"]

        # Create a single conversation (list of message dictionaries) for this example
        conversation = [
            {"role":"user","content":[
                {"type":"video","path":video_path},
                {"type":"text","text":question}]},
            {"role":"assistant","content":[{"type":"text","text":answer}]}
        ]
        all_conversations.append(conversation)

    if not all_conversations:
        # Return an empty dictionary if no valid conversations were constructed
        return {"input_ids": torch.tensor([]), "attention_mask": torch.tensor([]), "labels": torch.tensor([])}

    toks = tok.apply_chat_template(
        all_conversations, # Pass the list of conversations (List[List[Dict]])
        tokenize=True,
        padding=True,
        return_dict=True,
        return_tensors="pt"
    )
    return {"input_ids": toks["input_ids"],
            "attention_mask": toks["attention_mask"],
            "labels": toks["input_ids"]}

args = TrainingArguments(
    your_model_folder, per_device_train_batch_size=4,
    gradient_accumulation_steps=8, num_train_epochs=1, fp16=True,
    learning_rate=2e-4, lr_scheduler_type="cosine", warmup_ratio=0.03,
    logging_steps=10, save_total_limit=2,
    remove_unused_columns=False # Add this to prevent column removal by Trainer
)

Trainer(model=model, train_dataset=train_ds,
        data_collator=collate, args=args).train()
model.save_pretrained(your_model_folder)


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,1.519478
20,0.856389
30,0.745439
40,0.714947
50,0.736302
60,0.717740


In [ ]:
# We can load the adapter that way:
your_model_repo = '<yourUsername>/smolvlm2-vide-qlora-adapter'

base = AutoModelForImageTextToText.from_pretrained(
    model_id, dtype=torch.bfloat16, device_map="auto"
)
model = PeftModel.from_pretrained(base, your_model_folder)
model.push_to_hub(your_model_repo)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/489 [00:00<?, ?it/s]

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   3%|3         |  639kB / 20.0MB            

CommitInfo(commit_url='https://huggingface.co/mfarre/smolvlm2-vide-qlora-adapter/commit/3d9f416b721ba01ca0e2fa978114f705b053191a', commit_message='Upload model', commit_description='', oid='3d9f416b721ba01ca0e2fa978114f705b053191a', pr_url=None, repo_url=RepoUrl('https://huggingface.co/mfarre/smolvlm2-vide-qlora-adapter', endpoint='https://huggingface.co', repo_type='model', repo_id='mfarre/smolvlm2-vide-qlora-adapter'), pr_revision=None, pr_num=None)

In [ ]:
# Let's test it:

# We load the processor from the base model
processor = AutoProcessor.from_pretrained(model_id)

# We load a video
sample_video = hf_hub_download(
    repo_id="vlmbook/videos",
    repo_type="dataset",
    filename="MewNUHRGOm0.mp4",
)

# Auxiliary function to run inference
def ask_video(video_path: str, prompt: str, num_frames: int = 8) -> str:
    messages = [{
        "role": "user",
        "content": [
            {"type": "video", "path": video_path},
            {"type": "text",  "text": prompt},
        ],
    }]

    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
        do_sample_frames=True,
        num_frames=num_frames,
    )
    inputs = {k: (v.to(model.device) if torch.is_tensor(v) else v) for k, v in inputs.items()}
    if "pixel_values_videos" in inputs and torch.is_tensor(inputs["pixel_values_videos"]):
        inputs["pixel_values_videos"] = inputs["pixel_values_videos"].to(dtype=dtype)

    out = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=False,
    )

    prompt_len = inputs["input_ids"].shape[-1]
    return processor.batch_decode(out[:, prompt_len:], skip_special_tokens=True)[0].strip()


print(ask_video(sample_video, "What is happening here?", num_frames=8))

MewNUHRGOm0.mp4:   0%|          | 0.00/6.84M [00:00<?, ?B/s]

A group of people are working in a control room, possibly preparing for a space mission.
